In [68]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict
from pydantic import BaseModel, Field
import operator

In [69]:
load_dotenv()

True

In [70]:
model = ChatOpenAI(model='gpt-4o-mini')

In [71]:
class Evaluation_Schema(BaseModel):

    feedback: str = Field(description='Detailed feedback the essay')
    score: int = Field(description='Score out of 10', ge = 10, le = 10)

In [72]:
structured_model = model.with_structured_output(Evaluation_Schema)

In [73]:
essay = """The integration of Artificial Intelligence (AI) into web development marks a profound shift in how the internet is built, designed, and experienced. Historically, creating a web platform was a manual, labor-intensive craft requiring developers to write line-by-line code, manually design responsive layouts, and continuously debug complex logic. Today, AI acts as an accelerator, an intelligent co-pilot, and an architect, redefining every layer of the web development lifecycle—from back-end infrastructure to front-end user experience.  1. Automated Coding and Intelligent Co-PilotsOne of the most immediate impacts of AI on web development is the automation of code generation and optimization. Generative AI systems assist developers by:  Instant Code Generation: Producing boilerplate HTML, CSS, JavaScript, and complex database queries from plain text descriptions.Real-time Debugging & Refactoring: Identifying potential bugs, security vulnerabilities, and syntax errors before code hits production.  Predictive Autocomplete: Context-aware code completion that reduces repetitive typing and speeds up development cycles.Far from replacing human developers, these AI co-pilots streamline technical workflows, allowing engineers to shift their focus from mechanical syntax writing to higher-level architecture and problem-solving.  2. Dynamic UI/UX and Generative DesignTraditionally, web design relied heavily on static templates and fixed wireframes. AI is shifting this dynamic toward personalized, adaptive design systems:  Generative Wireframing: Converting rough sketches or natural language descriptions directly into responsive design prototypes.Algorithmic Personalization: Web interfaces that dynamically adapt layout, color schemes, and content structures in real time based on user preferences, device types, and browsing behavior.  Automated Accessibility (a11y): AI algorithms scan DOM structures to automatically generate alt text for visuals, correct color contrast issues, and optimize keyboard navigation for accessibility compliance."""

In [74]:
prompt = f'Evaluate the language quality of the following essay and provide a feedbacl and assign a score out of \10 {essay}'
structured_model.invoke(prompt)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your_ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

In [ ]:
class UPSCState(TypedDict):

    essay: str
    language_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float


In [ ]:
def evaluate_language(state: UPSCState):

    prompt = f'Evaluate the language quality of the following essay and provide a feedbacl and assign a score out of \10 {state["essay"]}'

    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}



In [ ]:
def evaluate_analysis(state: UPSCState):

    prompt = f'Evaluate the depth analysis of the following essay and provide a feedbacl and assign a score out of \10 {state["essay"]}'

    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}



In [ ]:
def evaluate_thought(state: UPSCState):

    prompt = f'Evaluate the thought of the following essay and provide a feedbacl and assign a score out of \10 {state["essay"]}'

    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}



In [ ]:
def final_thought(state: UPSCState):

    # summary Feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language feedback"]} \n depth of analysis feedback - {state["analysis feedback"]} \n clarity of thought feedback - {state["clarity feedback"]}'
    overall_feedback = model.invoke(prompt).content

    #avg Calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

    



In [ ]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('evaluate_final_Evaluation', evaluate_final_Evaluation)\


#edges
graph.add_edges(START, 'evaluate_language')    
graph.add_edges(START, 'evaluate_analysis')    
graph.add_edges(START, 'evaluate_thought')    

graph.add_edges('evaluate_language', 'evaluate_final_Evaluation')    
graph.add_edges('evaluate_analysis', 'evaluate_final_Evaluation')    
graph.add_edges('evaluate_thought', 'evaluate_final_Evaluation')  

graph.add_edge('evaluate_final_Evaluation', END)


graph.compile()

NameError: name 'Annotated' is not defined

In [75]:
initial_state = {
    'essay': essay
}

workflow.invoke(initial_state)

NameError: name 'workflow' is not defined